# Flow Matching Testing (DAC Embeddings)
Comprehensive testing for flow matching genre transformation using DAC embeddings (B, T, 768)

In [1]:
import torch
from models.flow import FlowMatching
from models.dit import DiT


film_conditioner.py STARTED


In [ ]:
# Test flow matching loss with DAC embeddings
B = 2      # batch size
T = 100    # time frames
D = 768    # DAC latent dimension

x0 = torch.randn(B, T, D)  # source (non-rock) DAC embeddings
x1 = torch.randn(B, T, D)  # target (rock) DAC embeddings
genre_ids = torch.tensor([1, 1])  # target = rock

dit = DiT(input_dim=D)
flow = FlowMatching(dit)

loss = flow.compute_loss(x0, x1, genre_ids)
print(f"Flow matching loss: {loss.item():.4f}")
print(f"✅ Loss computed successfully!")

In [ ]:
# Test velocity field prediction
with torch.no_grad():
    t = torch.rand(B)
    # For 3D tensors: expand t to (B, 1, 1) for broadcasting
    t_expanded = t.view(B, 1, 1)
    xt = (1 - t_expanded) * x0 + t_expanded * x1
    v_true = x1 - x0
    v_pred = dit(xt, t, genre_ids)

print(f"True velocity norm: {v_true.norm().item():.4f}")
print(f"Pred velocity norm: {v_pred.norm().item():.4f}")
print(f"Shape check: {v_pred.shape} == {v_true.shape}")

In [ ]:
# Overfit on single pair of DAC embeddings
optimizer = torch.optim.Adam(dit.parameters(), lr=1e-4)

losses = []
for step in range(200):
    optimizer.zero_grad()
    loss = flow.compute_loss(x0, x1, genre_ids)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if step % 20 == 0:
        print(f"Step {step:4d} → loss: {loss.item():.4f}")

print(f"\n✅ Loss reduced from {losses[0]:.4f} to {losses[-1]:.4f}")

In [ ]:
# Test Euler sampling with DAC embeddings
with torch.no_grad():
    x_gen = flow.sample_euler(x0, genre_ids=1, num_steps=50)

dist_start = torch.mean(torch.abs(x0 - x1))
dist_end = torch.mean(torch.abs(x_gen - x1))

print(f"Distance before transformation: {dist_start.item():.4f}")
print(f"Distance after transformation:  {dist_end.item():.4f}")
print(f"Output shape: {x_gen.shape}")

In [ ]:
# Compare Euler vs Heun sampling (Heun is 2nd order, should be more accurate)
with torch.no_grad():
    x_euler = flow.sample_euler(x0, 1, num_steps=30)
    x_heun = flow.sample_heun(x0, 1, num_steps=30)

diff = torch.mean(torch.abs(x_euler - x_heun))
print(f"Euler vs Heun difference: {diff.item():.4f}")
print(f"Euler output shape: {x_euler.shape}")
print(f"Heun output shape:  {x_heun.shape}")

In [ ]:
# Load real DAC embeddings (if available)
import torch
from pathlib import Path

dac_path = Path('../data/output/non_rock_dac')
if dac_path.exists():
    dac_files = list(dac_path.glob('*.pt'))
    if dac_files:
        data = torch.load(dac_files[0])
        print(f"DAC file keys: {list(data.keys())}")
        emb = data['embeddings']
        print(f"Embeddings shape: {emb.shape}")
        print(f"Latent dim: {emb.shape[-1]}")
else:
    print(f"DAC directory not found: {dac_path}")
    print("Run audio-to-DAC conversion first using NeuralCodecConverter")

Shape: torch.Size([1, 100, 2811])


In [ ]:
# Full integration test with config
import torch
from models.dit import DiT
from models.flow import FlowMatching
from omegaconf import OmegaConf

# Load config
config = OmegaConf.load('../configs/dit.yaml')

# Initialize model with config for DAC
model = DiT(
    input_dim=config.dit_model.input_dim,   # 768 for DAC
    embed_dim=config.dit_model.embed_dim,
    num_blocks=config.dit_model.num_blocks,
    num_heads=config.dit_model.num_heads,
    num_genres=config.dit_model.num_genres,
    hidden_dim=config.dit_model.hidden_dim,
    dropout=config.dit_model.dropout,
)

flow = FlowMatching(model)

# Create fake DAC embeddings
B, T, D = 2, 100, config.dit_model.input_dim
x0 = torch.randn(B, T, D)  # source
x1 = torch.randn(B, T, D)  # target
genre_ids = torch.tensor([1, 1])

# Test forward pass
loss = flow.compute_loss(x0, x1, genre_ids)
print(f"Input shape:  {x0.shape} (B, T, latent_dim)")
print(f"Loss:         {loss.item():.4f}")

# Test sampling
with torch.no_grad():
    x_gen = flow.sample_euler(x0, genre_ids=1, num_steps=10)
print(f"Output shape: {x_gen.shape}")
print(f"\n✅ Integration test passed!" if x_gen.shape == x0.shape else "❌ Shape mismatch!")

Input shape: torch.Size([1, 1, 100, 2816])
Output shape: torch.Size([1, 1, 100, 2816])
✅ Dimension test passed!
